## Imports and Installations

In [1]:
import os
temp_path = os.getcwd().split('\\')
project_dir = '\\'.join(temp_path[:temp_path.index('SOURCE') + 1])

In [2]:
import sys

sys.path.insert(0, f'{project_dir}/code/Packages')

In [3]:
!pip install igraph

In [4]:
import os
import pandas as pd
import igraph as ig
from tqdm import tqdm
from glob import glob 

⚠⚠⚠

In [5]:
import preprocessing

In [11]:
preprocessing.set_export_folder("final_edgelists") # As of April 5, 2022, do not change!!

Export directory set to C:\Users\ASUS\Downloads\SOURCE/data/3 - Network Generation/final_edgelists.


In [12]:
tqdm.pandas()

---

In [13]:
network_types = ['temporal', 'aggregated']
network_type = network_types[1] # TODO: change
print(f"Selected network type: {network_type}")

Selected network type: aggregated


## Create Weighted Edge Lists

In [14]:
directory = f"{project_dir}/data/3 - Network Generation/{preprocessing.export_folder_name}"

In [15]:
path = f'{directory}'

edgelists = []
for category in sorted(os.listdir(path)):
    if category not in ["nodelist.pkl", "[backup copy] of nodelist.pkl"]:
        print(f'Collecting {category}...')
        for source in sorted(os.listdir(f'{path}/{category}')):
            print(f'Collecting {source}...')
            years = sorted(os.listdir(f'{path}/{category}/{source}'))

            for year in years:    
                months = sorted(os.listdir(f'{path}/{category}/{source}/{year}'))
                print(f"{year} - ", end="")

                for month in months:
                    file_path = f'{path}/{category}/{source}/{year}/{month}/'

                    csv_files = sorted(glob(f'{file_path}/*[!_cleaned].csv'))

                    print(f"{month}, ", end="")

                    # loop through the files and read them in with pandas
                    for edgelist in csv_files:
                        if network_type == 'temporal':
                            edgelists.append(pd.read_csv(open(edgelist, encoding="utf8"), usecols=['word1', 'word2', 'year', 'source', 'source_type']))
                        if network_type == 'aggregated':
                            edgelists.append(pd.read_csv(open(edgelist, encoding="utf8"), usecols=['word1', 'word2']))  
                print()
            print()

2010 - 12, 
2011 - 11, 12, 4, 5, 8, 9, 
2012 - 1, 10, 12, 2, 4, 6, 7, 8, 
2014 - 1, 10, 11, 12, 2, 3, 4, 5, 6, 7, 8, 9, 
2015 - 1, 10, 11, 12, 2, 3, 4, 5, 6, 7, 8, 9, 
2016 - 1, 10, 11, 12, 2, 3, 4, 5, 6, 7, 8, 9, 
2017 - 1, 10, 11, 12, 2, 3, 4, 5, 6, 7, 8, 9, 
2018 - 1, 10, 11, 12, 2, 3, 4, 5, 6, 7, 8, 9, 
2019 - 1, 10, 11, 12, 2, 3, 4, 5, 6, 7, 8, 9, 
2020 - 1, 10, 11, 12, 2, 3, 4, 5, 6, 7, 
2021 - 10, 



In [16]:
print("Merging to single edgelist...")           
edgelist_df = pd.concat(edgelists, ignore_index=True)

Merging to single edgelist...


In [17]:
print("Calculating weighted edgelist...")           
weighted_edgelist = edgelist_df.groupby(edgelist_df.columns.tolist(), as_index = False).size().rename(columns={"size": "weight"})

Calculating weighted edgelist...


In [18]:
weighted_edgelist

,word1,word2,weight
0,0,1,1
1,0,21,1
2,0,290,1
3,0,548,1
4,0,733,1
...,...,...,...
390240,37595,28758,1
390241,37595,34116,1
390242,37595,37594,1
390243,37595,37596,1


---

## Create igraph Object

In [19]:
weighted_edgelist['word1'] = weighted_edgelist['word1'].to_numpy('int32')

In [20]:
weighted_edgelist['word2'] = weighted_edgelist['word2'].to_numpy('int32')

In [21]:
graph = ig.Graph.DataFrame(weighted_edgelist, directed=False, use_vids=True)

In [22]:
graph = graph.simplify(multiple=False)

### Add node attributes

In [23]:
graph.vs['index'] = [vertex.index for vertex in graph.vs]

In [24]:
inverted_nodelist_dict = preprocessing.get_inverted_nodelist()

In [25]:
for vertex in graph.vs:
    vertex['word'] = inverted_nodelist_dict[vertex['index']]

In [26]:
directory = f"{project_dir}/data/3 - Network Generation"

if network_type == 'temporal':
    graph.write(f"{directory}/temporal_multiplex_network.graphml", format="graphml")
if network_type == 'aggregated':
    graph.write(f"{directory}/aggregated_network.graphml", format="graphml")